# RAG notebook

Prerequisites before running all cells:
- `os_book.pdf` exists in the repository root.
- `.env` contains `OPENAI_API_KEY` and `PINECONE_API_KEY`.
- Required packages from `requirements.txt` are installed.


In [1]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader


In [2]:
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "rag-llm-index")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")
PDF_PATH = Path(os.getenv("PDF_PATH", "angular.pdf"))

if not OPENAI_API_KEY:
    raise ValueError("Missing OPENAI_API_KEY in environment or .env")

if not PINECONE_API_KEY:
    raise ValueError("Missing PINECONE_API_KEY in environment or .env")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF file not found: {PDF_PATH.resolve()}")


In [3]:
def read_pdf(file_path: Path) -> str:
    reader = PdfReader(str(file_path))
    pages = [(page.extract_text() or "") for page in reader.pages]
    text = "\n".join(pages).strip()
    if not text:
        raise ValueError(f"No extractable text found in {file_path}")
    return text


def split_text_into_chunks(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_text(text)


In [4]:
openai_embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
client = OpenAI(api_key=OPENAI_API_KEY)


def generate_embeddings(text_chunks: list[str]) -> list[list[float]]:
    return openai_embeddings.embed_documents(text_chunks)


In [5]:
pc = Pinecone(api_key=PINECONE_API_KEY)
spec = ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION)

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]
if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric="dotproduct",
        spec=spec,
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)

index = pc.Index(PINECONE_INDEX_NAME)


In [6]:
def save_embeddings_to_pinecone(
    embeddings: list[list[float]],
    metadata: list[dict],
    namespace: str | None = None,
    batch_size: int = 50,
):
    total_uploaded = 0

    try:
        for start in range(0, len(embeddings), batch_size):
            end = min(start + batch_size, len(embeddings))
            vectors = [
                {
                    "id": f"id-{i}",
                    "values": embeddings[i],
                    "metadata": metadata[i],
                }
                for i in range(start, end)
            ]

            if namespace:
                index.upsert(vectors=vectors, namespace=namespace)
            else:
                index.upsert(vectors=vectors)

            total_uploaded += len(vectors)
            print(f"Uploaded batch {start}-{end - 1}")

        return total_uploaded
    except Exception as exc:
        raise RuntimeError(
            "Failed to upsert vectors to Pinecone. The request may be too large, the network may be blocked, or the API settings may be invalid."
        ) from exc


In [7]:
text = read_pdf(PDF_PATH)
text_chunks = split_text_into_chunks(text)
metadata = [
    {
        "chunk_id": i,
        "source": PDF_PATH.name,
        "text": text_chunks[i],
    }
    for i in range(len(text_chunks))
]

print(f"Loaded {len(text_chunks)} chunks from {PDF_PATH.name}")


Loaded 834 chunks from os_book.pdf


In [8]:
def query_pinecone_with_text(query_text: str, top_k: int = 5, namespace: str | None = None):
    query_embedding = openai_embeddings.embed_query(query_text)
    kwargs = {
        "vector": query_embedding,
        "top_k": top_k,
        "include_metadata": True,
    }
    if namespace:
        kwargs["namespace"] = namespace
    return index.query(**kwargs)


def print_matches(results) -> None:
    for match in results.get("matches", []):
        print("Score:", match.get("score"))
        print("Chunk ID:", match.get("id"))
        print("Source PDF:", match.get("metadata", {}).get("source"))
        print("Text:", match.get("metadata", {}).get("text"))
        print("-" * 29)


In [9]:
def generate_text(query_text: str, search_results) -> str:
    matches = search_results.get("matches", [])
    if not matches:
        return "No relevant chunks were found in Pinecone."

    context = "\n".join(match["metadata"]["text"] for match in matches if "metadata" in match)
    prompt = (
        f"Given the question: {query_text}\n\n"
        f"Use only the following document context:\n{context}\n\n"
        "Provide a comprehensive answer. If the context is insufficient, say so clearly."
    )
    completion = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You answer questions using only the supplied document context."},
            {"role": "user", "content": prompt},
        ],
    )
    return completion.choices[0].message.content


In [10]:
query = "what is mutex?"
results = query_pinecone_with_text(query)
print_matches(results)
print(generate_text(query, results))


Score: 0.799255371
Chunk ID: id-458
Source PDF: os_book.pdf
Text: גלובלי. המנגנון השני הוא  Mutex, שמשמש לסנכרון ביןProcessים. מנגנון זה מצריך שימוש במשאבי  
מערכת ההפעלה וקריאה לפונקציות שפועלות עם Mutexים דורשת לכן פניה אל הקרנל. הפניה אל הקרנל
דורשת זמן רב יותר מאשר פניה אל משתנה גלובלי כמו CriticalSection, וזו הסיבה לכך שבתרגיל
הפילוסופים השימוש ב-CriticalSections נתן מדידת זמן קצרה משמעותית 
לבסוף למדנו  על מנגנון האיתות Semaphore, שמאפשר להקצות למשאב כמות מקסימלית של משתמשים  
שחולקים אותו. 
  
פרק6 – זיכרון 
 
118 
פרק 6 – זיכרון
-----------------------------
Score: 0.798400879
Chunk ID: id-426
Source PDF: os_book.pdf
Text: למטה, מוקף לצורך ההמחשה במסגרת שחורה, ניתן למצוא את שם ה-Mutex  שה-Process של כרום
משתמש בו. מסיבות היסטוריות, הקרנל של  Windows  קוראMutant ל-Mutex, זה אותו דבר.
-----------------------------
Score: 0.787597656
Chunk ID: id-448
Source PDF: os_book.pdf
Text: הגדרת ה-Mutexים מתבצעת על ידי הפונקציהCreateMutex, שמקבלת בפשטות את השם שבחרתם עבור
כל Mutex. העזרו בת